# Camera Pose Optimization From Real Background-Run Data -- All Frames

Batch version of [`CameraPoseOptimizationFromRealData.ipynb`](CameraPoseOptimizationFromRealData.ipynb):
same single-bubble event, same render-at-reco-position sanity check, but looped over
*every* frame of that event with a valid 3D reconstruction and pixel track, instead of
just one. Every rendered frame gets its own saved real-vs-rendered comparison image.

**Runtime note:** unlike the single-frame notebook, the per-frame render cost here is not
a one-time JIT-warmup cost -- each frame is a genuinely separate CPU path-traced render.
At full quality (1280x800, 1024 samples) that measured ~300s/frame with no GPU, so a full
event (~30 frames) is a couple of hours. The quality toggle below defaults to a fast, low
sample-count/resolution pass for iterating on the pipeline; switch it off for real
(publication-quality) renders once you know the pipeline works.

In [1]:
import os, sys
if not os.path.isdir("modules"):
    os.chdir("..")
sys.path.insert(0, "modules")

from utils import *
import real_data_loading as rdl
import pandas as pd

In [2]:
# Group every real-data render for a given event under
# outputs/<run>_ev<ev>/cam<camera>/frame<NN>/ instead of flat in outputs/<date>/, so a
# whole event's renders are easy to browse together. This still goes through render()'s
# own save_mono/save_rgb (same file-naming convention every notebook in this repo uses)
# -- it just relocates the two files it wrote right after it writes them.
def event_output_dir(run_name, ev, camera, frame):
    d = pathlib.Path(output_dir) / f"{run_name}_ev{ev}" / f"cam{camera}" / f"frame{frame:02d}"
    d.mkdir(parents=True, exist_ok=True)
    return d


def move_latest_render_outputs(tag, dest_dir, retries=20, delay=0.5):
    """Moves the mono/rgb files render() just wrote (matched by their unique
    save_string_append tag) out of outputs/<date>/ and into dest_dir. render()'s own
    'Saved Image' print can appear slightly before the file is actually flushed to disk
    (seen in practice), so this polls briefly instead of assuming it's already there."""
    date_dir = pathlib.Path(output_dir) / time.strftime('%Y_%m_%d')
    moved = {}
    for kind in ("mono", "rgb"):
        matches = []
        for _ in range(retries):
            matches = sorted(date_dir.glob(f"*_{kind}_{tag}.png"))
            if matches:
                break
            time.sleep(delay)
        if matches:
            dest = dest_dir / matches[-1].name
            matches[-1].rename(dest)
            moved[kind] = dest
        else:
            print(f"WARNING: no {kind} file found for tag {tag!r} after {retries * delay:.1f}s")
    try:
        date_dir.rmdir()  # tidy up render()'s now-empty outputs/<date>/ scratch dir
    except OSError:
        pass  # not empty (other renders landed here too) -- leave it alone
    return moved

In [3]:
# Pick a run and a clean single-bubble event (handscanned nbub == 1, crosshairs good),
# and load its per-camera pixel track and triangulated 3D reconstruction -- same choice
# as the single-frame notebook, for consistency.
run_name = "20251114_37"
run_dir = rdl.RUN_DIRS[run_name]
camera = 1

ev = rdl.single_bubble_events(run_dir)[0]
tracks, t0_info = rdl.load_single_bubble_track(run_dir, ev)
reco = rdl.load_reco_single(run_dir, ev)

print(f"run {run_name}, event {ev}")
print(f"cameras with a track: {list(tracks.keys())}")
print(f"t0: {t0_info}")

run 20251114_37, event 1
cameras with a track: [1, 2, 3]
t0: {'frame': 4, 'cams': (2, 3)}


In [4]:
# Every frame with both a valid 3D reconstruction and a pixel detection in cam{camera}
# -- not just one, unlike the single-frame notebook.
reco_by_frame = {r["frame"]: r for r in reco if r["coord"] is not None}
track_by_frame = {row["frame"]: row for row in tracks[camera]}
valid_frames = sorted(set(reco_by_frame) & set(track_by_frame))

print(f"{len(valid_frames)} valid frames for cam{camera}: {valid_frames}")

29 valid frames for cam1: [16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44]


In [5]:
# Toggle for quick, low-quality test renders vs. full-quality ones. Render cost scales
# roughly with width * height * sample_count, so quick mode is dramatically faster --
# useful for checking the pipeline/positions look right before committing to a full,
# slow run over every frame. Flip to False for real (publication-quality) renders.
QUICK_TEST = True

if QUICK_TEST:
    render_width, render_height, sample_count = 640, 400, 64
else:
    render_width, render_height, sample_count = 1280, 800, 1024

quality_tag = "quick" if QUICK_TEST else "full"
print(f"QUICK_TEST={QUICK_TEST} -> {render_width}x{render_height} @ {sample_count} spp ({quality_tag})")

QUICK_TEST=True -> 640x400 @ 64 spp (quick)


In [6]:
# Set up the chamber scene once -- only the bubble's placement changes per frame below.
materials = create_materials(with_fluids=True)
components = load_components(materials=materials, use_distorted_jar=True)
sensor = create_sensor(sensor_number=camera, sample_count=sample_count,
                       width=render_width, height=render_height)
camera_origin_cm = np.array(sensor["to_world"].matrix)[:3, 3]

# The real detector images are 1280x800 regardless of our render-quality toggle above --
# this converts a *real* pixel radius to a physical size, so it must stay tied to the
# real camera's own resolution/fov, not whatever resolution we choose to render at.
focal_length_px = get_ideal_camera_matrix(fov=100, fov_axis='x', image_width=1280, image_height=800)[0, 0]

2026-09-14 11:06:24 WARN  [PLYMesh] "outer_jar_outer_surface_top_distorted.ply": skipping unknown element "edge"


2026-09-14 11:06:24 WARN  [PLYMesh] "outer_jar_inner_surface_top_distorted.ply": skipping unknown element "edge"
2026-09-14 11:06:24 WARN  [PLYMesh] "inner_jar_outer_surface_bottom.ply": skipping unknown element "edge"


In [7]:
# Render every valid frame's reconstructed bubble position and save a real-vs-rendered
# comparison image for *each* one (not just a subset) -- same steps as the single-frame
# notebook, just looped, reusing the chamber/materials/sensor built above. Every frame's
# outputs land in their own outputs/<run>_ev<ev>/cam<camera>/frame<NN>/ folder.
results = []
for frame in valid_frames:
    row = track_by_frame[frame]
    pixel_pos = row["pos"]
    pixel_radius = row["rad"]

    coord_mm = np.array(reco_by_frame[frame]["coord"])
    location_cm = coord_mm / 10.0
    distance_cm = np.linalg.norm(location_cm - camera_origin_cm)
    bubble_radius_cm = pixel_radius * distance_cm / focal_length_px

    components.update({
        'bubble': mi.load_dict({
            'type': 'sphere',
            'focused-emitter': {
                'type': 'area',
                'radiance': {'type': 'spectrum', 'value': 60.0},
            },
            'to_world': mi.ScalarTransform4f.translate(location_cm).scale(float(bubble_radius_cm)),
        })
    })
    scene = load_scene(components=components, sensor=sensor)

    tag = f"{run_name}_ev{ev}_frame{frame}_cam{camera}_{quality_tag}"
    image = render(scene=scene, denoise=False, save_mono=True, save_rgb=True, save_string_append=tag)

    dest_dir = event_output_dir(run_name, ev, camera, frame)
    move_latest_render_outputs(tag, dest_dir)

    real_image = load_image(rdl.camera_image_path(run_dir, ev, camera, frame))

    fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
    axes[0].imshow(real_image, cmap='gray')
    axes[0].set_title(f"Real cam{camera}, ev {ev} frame {frame}")
    axes[1].imshow(np.clip(image ** (1 / 2.2), 0, 1))
    axes[1].set_title(f"Rendered ({quality_tag}), frame {frame}")
    for ax in axes:
        ax.axis('off')

    compare_path = dest_dir / f"{time.strftime('%H_%M_%S')}_comparison_{tag}.png"
    fig.savefig(compare_path, dpi=120, bbox_inches='tight')
    plt.close(fig)

    results.append({
        "frame": frame,
        "pixel_pos": pixel_pos,
        "pixel_radius_px": pixel_radius,
        "location_cm": tuple(location_cm),
        "distance_cm": distance_cm,
        "bubble_radius_cm": bubble_radius_cm,
        "comparison_path": str(compare_path),
    })
    print(f"frame {frame}: pixel {pixel_pos}, {pixel_radius}px -> {bubble_radius_cm:.3f}cm bubble, "
          f"saved {compare_path.name}")

Saved Image: ./outputs/2026_09_14/11_06_25_mono_20251114_37_ev1_frame16_cam1_quick.png
Saved Image: ./outputs/2026_09_14/11_06_25_rgb_20251114_37_ev1_frame16_cam1_quick.png


frame 16: pixel (515.0, 448.0), 7.0px -> 0.492cm bubble, saved 11_06_30_comparison_20251114_37_ev1_frame16_cam1_quick.png


Saved Image: ./outputs/2026_09_14/11_06_31_mono_20251114_37_ev1_frame17_cam1_quick.png
Saved Image: ./outputs/2026_09_14/11_06_31_rgb_20251114_37_ev1_frame17_cam1_quick.png
frame 17: pixel (512.0, 449.0), 21.0px -> 1.543cm bubble, saved 11_06_36_comparison_20251114_37_ev1_frame17_cam1_quick.png


Saved Image: ./outputs/2026_09_14/11_06_37_mono_20251114_37_ev1_frame18_cam1_quick.png
Saved Image: ./outputs/2026_09_14/11_06_37_rgb_20251114_37_ev1_frame18_cam1_quick.png
frame 18: pixel (513.0, 444.0), 13.0px -> 0.952cm bubble, saved 11_06_43_comparison_20251114_37_ev1_frame18_cam1_quick.png


Saved Image: ./outputs/2026_09_14/11_06_44_mono_20251114_37_ev1_frame19_cam1_quick.png
Saved Image: ./outputs/2026_09_14/11_06_44_rgb_20251114_37_ev1_frame19_cam1_quick.png


...

Saved Image: ./outputs/2026_09_14/11_08_54_mono_20251114_37_ev1_frame40_cam1_quick.png
Saved Image: ./outputs/2026_09_14/11_08_54_rgb_20251114_37_ev1_frame40_cam1_quick.png
frame 40: pixel (521.0, 454.0), 14.0px -> 0.941cm bubble, saved 11_08_59_comparison_20251114_37_ev1_frame40_cam1_quick.png


Saved Image: ./outputs/2026_09_14/11_09_00_mono_20251114_37_ev1_frame41_cam1_quick.png
Saved Image: ./outputs/2026_09_14/11_09_00_rgb_20251114_37_ev1_frame41_cam1_quick.png
frame 41: pixel (523.0, 451.0), 17.0px -> 1.130cm bubble, saved 11_09_05_comparison_20251114_37_ev1_frame41_cam1_quick.png


Saved Image: ./outputs/2026_09_14/11_09_06_mono_20251114_37_ev1_frame42_cam1_quick.png
Saved Image: ./outputs/2026_09_14/11_09_06_rgb_20251114_37_ev1_frame42_cam1_quick.png
frame 42: pixel (523.0, 427.0), 41.0px -> 2.743cm bubble, saved 11_09_11_comparison_20251114_37_ev1_frame42_cam1_quick.png


Saved Image: ./outputs/2026_09_14/11_09_12_mono_20251114_37_ev1_frame43_cam1_quick.png
Saved Image: ./outputs/2026_09_14/11_09_12_rgb_20251114_37_ev1_frame43_cam1_quick.png
frame 43: pixel (522.0, 432.0), 37.0px -> 2.446cm bubble, saved 11_09_17_comparison_20251114_37_ev1_frame43_cam1_quick.png


Saved Image: ./outputs/2026_09_14/11_09_18_mono_20251114_37_ev1_frame44_cam1_quick.png
Saved Image: ./outputs/2026_09_14/11_09_18_rgb_20251114_37_ev1_frame44_cam1_quick.png
frame 44: pixel (525.0, 446.0), 25.0px -> 1.586cm bubble, saved 11_09_24_comparison_20251114_37_ev1_frame44_cam1_quick.png


In [8]:
# One row per processed frame, with a saved comparison image for every single one --
# open comparison_path to see the full-resolution real-vs-rendered pair for that frame.
results_df = pd.DataFrame(results)
results_df

,frame,pixel_pos,pixel_radius_px,location_cm,distance_cm,bubble_radius_cm,comparison_path
0,16,"(515.0, 448.0)",7.0,"(-4.502903293098727, 1.6315005806825316, -17.9...",37.769160,0.492314,outputs/20251114_37_ev1/cam1/frame16/11_06_30_...
1,17,"(512.0, 449.0)",21.0,"(-4.760256127410914, 1.850887823896064, -19.65...",39.448115,1.542595,outputs/20251114_37_ev1/cam1/frame17/11_06_36_...
2,18,"(513.0, 444.0)",13.0,"(-4.634654708896951, 1.8222704285739368, -19.5...",39.336428,0.952236,outputs/20251114_37_ev1/cam1/frame18/11_06_43_...
3,19,"(509.0, 447.0)",4.0,"(-4.9111123253234155, 1.947048391698506, -20.0...",39.818759,0.296588,outputs/20251114_37_ev1/cam1/frame19/11_06_49_...
4,20,"(506.0, 453.0)",3.0,"(-5.066110545576427, 1.9697741458334668, -19.8...",39.702786,0.221793,outputs/20251114_37_ev1/cam1/frame20/11_06_55_...
5,21,"(512.0, 451.0)",5.0,"(-4.810971412455171, 1.7585669672489352, -18.8...",38.660023,0.359947,outputs/20251114_37_ev1/cam1/frame21/11_07_01_...
6,22,"(520.0, 457.0)",7.0,"(-4.726177772074374, 1.5472613714077674, -17.4...",37.234134,0.485340,outputs/20251114_37_ev1/cam1/frame22/11_07_07_...
7,23,"(520.0, 456.0)",15.0,"(-4.580299027504411, 1.155025394942428, -18.53...",38.179733,1.066426,outputs/20251114_37_ev1/cam1/frame23/11_07_14_...
8,24,"(520.0, 456.0)",6.0,"(-4.564847452533344, 1.6930221116336626, -17.4...",37.359969,0.417411,outputs/20251114_37_ev1/cam1/frame24/11_07_20_...
9,25,"(511.0, 454.0)",5.0,"(-4.79176383198893, 1.9795026582855928, -18.19...",38.105062,0.354780,outputs/20251114_37_ev1/cam1/frame25/11_07_26_...
